# Causal Uplift Modeling — EBM T-Learner
**Members 3, 4, 5** — Modelling · Explainability · Evaluation

Run cells top to bottom.  Each section is clearly marked with its owner.

## 0 · Setup — run this first

In [3]:
# Add the project root to Python path so we can import from src/
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# Install any missing packages
# !pip install interpret xgboost scikit-uplift shap matplotlib --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.calibration import calibration_curve
from interpret.glassbox import ExplainableBoostingClassifier
import xgboost as xgb

from src.config import GLOBAL_SEED, MODELS_DIR, RESULTS_DIR, CV_FOLDS
from src.data_loader import load_and_preprocess_hillstrom, check_propensity_overlap
from src.custom_tlearner import CustomTLearner

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup complete.')

Setup complete.


## 1 · Load data  (Member 1 pipeline)

In [4]:
# This single call runs: load → validate → encode → stratified split
X_train, X_test, y_train, y_test, t_train, t_test = load_and_preprocess_hillstrom()

# Save feature names — needed for SHAP plots later
feature_names = list(X_train.columns)
print(f'Features ({len(feature_names)}):', feature_names)

[1/5] Loading data from: /home/shaurya/Causal-Uplift-EBM/src/../data/hillstrom.csv
      Shape: (64000, 12)
      No missing values found. Good.
      Treatment distribution:
treatment
1    42694
0    21306
Name: count, dtype: int64
      Target ('visit') distribution:
visit
0    54606
1     9394
Name: count, dtype: int64
[2/5] Engineering features...
      Feature matrix columns after encoding: ['recency', 'history_segment', 'history', 'mens', 'womens', 'newbie', 'segment', 'visit', 'conversion', 'spend', 'treatment', 'zip_code_Surburban', 'zip_code_Urban', 'channel_Phone', 'channel_Web']
[5/5] Saving processed data to: /home/shaurya/Causal-Uplift-EBM/src/../data/hillstrom_processed.csv
      Done.
[3/5] Creating stratified 80/20 train/test split...
      Train size: 51,200  |  Test size: 12,800
      Train treatment rate: 0.667  |  Test: 0.667
      Train target rate:    0.147  |  Test: 0.147

=== Data pipeline complete ===
  X_train shape : (51200, 11)
  X_test shape  : (12800, 11)


In [6]:
# Propensity overlap check — proves the RCT assumption holds
# Expected: two overlapping humps centred near 0.33 (1/3 in treatment group)
propensity_scores = check_propensity_overlap(
    X_train, t_train,
    save_path=os.path.join(RESULTS_DIR, 'propensity_overlap.png')
)

[4/5] Running propensity score overlap check...


ValueError: could not convert string to float: '5) $500 - $750'

## 2 · Member 3 — Model optimisation
We tune BOTH XGBoost (baseline) and EBM (champion) using GridSearchCV.
Crucially, we use **StratifiedKFold** because the positive class rate
is ~15% for `visit` — plain KFold would create folds with zero positives.

In [ ]:
# ============================================================
# Shared CV strategy — used for EVERY GridSearchCV call below
# ============================================================
# StratifiedKFold preserves the class ratio in each fold.
# shuffle=True prevents any accidental ordering bias in the data.
# random_state=GLOBAL_SEED ensures the same folds every run.
cv_strategy = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=GLOBAL_SEED
)

In [ ]:
# ============================================================
# 2a. Split training data by treatment group
# ============================================================
# The T-Learner requires SEPARATE models for treated and control.
# We tune each model ONLY on its relevant subgroup —
# not on the whole training set.

treated_mask = (t_train == 1)
control_mask = (t_train == 0)

X_treated = X_train[treated_mask]
y_treated = y_train[treated_mask]

X_control = X_train[control_mask]
y_control = y_train[control_mask]

print(f'Treated group size : {len(X_treated):,}')
print(f'Control group size : {len(X_control):,}')

In [ ]:
# ============================================================
# 2b. XGBoost tuning  (BASELINE model)
# ============================================================
# XGBoost is the industry gold standard for tabular data.
# We use it as a performance ceiling: if EBM matches it,
# we've achieved interpretability WITHOUT sacrificing accuracy.

xgb_param_grid = {
    # How fast the model learns — smaller = more trees needed but
    # less chance of overfitting any single tree's noise
    'learning_rate': [0.01, 0.1, 0.2],

    # Max depth of each decision tree — deeper = more complex
    # patterns, also more risk of overfitting
    'max_depth': [3, 5, 7],

    # Number of trees to build — more trees = more capacity
    'n_estimators': [100, 300],

    # Fraction of training rows used per tree — adds randomness
    # which acts as a regulariser
    'subsample': [0.8, 1.0],
}

# ROC-AUC is the right metric here — we care about ranking users
# by conversion probability, not raw accuracy (which is misleading
# on imbalanced classes).
xgb_grid_treated = GridSearchCV(
    estimator=xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        use_label_encoder=False,
        random_state=GLOBAL_SEED,
        verbosity=0
    ),
    param_grid=xgb_param_grid,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1,       # use all CPU cores
    verbose=1
)

print('Tuning XGBoost on TREATED group...')
xgb_grid_treated.fit(X_treated, y_treated)
optimal_xgb_treatment = xgb_grid_treated.best_estimator_
print(f'Best XGB treated params : {xgb_grid_treated.best_params_}')
print(f'Best XGB treated AUC    : {xgb_grid_treated.best_score_:.4f}')

In [ ]:
# Repeat for CONTROL group
xgb_grid_control = GridSearchCV(
    estimator=xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        use_label_encoder=False,
        random_state=GLOBAL_SEED,
        verbosity=0
    ),
    param_grid=xgb_param_grid,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

print('Tuning XGBoost on CONTROL group...')
xgb_grid_control.fit(X_control, y_control)
optimal_xgb_control = xgb_grid_control.best_estimator_
print(f'Best XGB control params : {xgb_grid_control.best_params_}')
print(f'Best XGB control AUC    : {xgb_grid_control.best_score_:.4f}')

# Save tuning logs for the report
pd.DataFrame(xgb_grid_treated.cv_results_).to_csv(os.path.join(RESULTS_DIR, 'xgb_treated_cv_results.csv'), index=False)
pd.DataFrame(xgb_grid_control.cv_results_).to_csv(os.path.join(RESULTS_DIR, 'xgb_control_cv_results.csv'), index=False)

In [ ]:
# ============================================================
# 2c. EBM tuning  (CHAMPION model)
# ============================================================
# The Explainable Boosting Machine is a Generalized Additive Model:
#   g(E[y]) = β₀ + Σ fᵢ(xᵢ) + Σ fᵢⱼ(xᵢ, xⱼ)
# Each fᵢ is a learned step-function over one feature.
# fᵢⱼ captures pairwise interactions between two features.
# This makes EBM as accurate as gradient boosting but
# natively interpretable — you can PLOT each fᵢ directly.

ebm_param_grid = {
    # Step size for gradient boosting — smaller = finer shape functions
    'learning_rate': [0.01, 0.05, 0.1],

    # Histogram bins for each feature — more bins = smoother f_i curves
    # at the cost of more memory and computation
    'max_bins': [128, 256],

    # Number of pairwise interaction terms fᵢⱼ to learn
    # More interactions → higher accuracy, less interpretability
    'interactions': [5, 10],

    # Minimum samples in a leaf — key regularisation parameter
    # Higher value → smoother model, less overfitting
    # CRITICAL on small subgroups (treated subset is ~21K rows)
    'min_samples_leaf': [2, 5],

    # Number of boosting rounds per feature — more rounds = more fitted
    # step-function detail. Cap this to avoid overfitting.
    'max_rounds': [5000, 8000],
}

ebm_grid_treated = GridSearchCV(
    estimator=ExplainableBoostingClassifier(random_state=GLOBAL_SEED),
    param_grid=ebm_param_grid,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

print('Tuning EBM on TREATED group (this takes a few minutes)...')
ebm_grid_treated.fit(X_treated, y_treated)
optimal_ebm_treatment = ebm_grid_treated.best_estimator_
print(f'Best EBM treated params : {ebm_grid_treated.best_params_}')
print(f'Best EBM treated AUC    : {ebm_grid_treated.best_score_:.4f}')

In [ ]:
ebm_grid_control = GridSearchCV(
    estimator=ExplainableBoostingClassifier(random_state=GLOBAL_SEED),
    param_grid=ebm_param_grid,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

print('Tuning EBM on CONTROL group...')
ebm_grid_control.fit(X_control, y_control)
optimal_ebm_control = ebm_grid_control.best_estimator_
print(f'Best EBM control params : {ebm_grid_control.best_params_}')
print(f'Best EBM control AUC    : {ebm_grid_control.best_score_:.4f}')

pd.DataFrame(ebm_grid_treated.cv_results_).to_csv(os.path.join(RESULTS_DIR, 'ebm_treated_cv_results.csv'), index=False)
pd.DataFrame(ebm_grid_control.cv_results_).to_csv(os.path.join(RESULTS_DIR, 'ebm_control_cv_results.csv'), index=False)

In [ ]:
# ============================================================
# 2d. Build T-Learners and fit on full training set
# ============================================================
# Now inject the optimised base models into our custom T-Learner.
# The T-Learner's fit() will handle the treatment/control routing
# internally — we just pass the full training data.

tlearner_xgb = CustomTLearner(
    treatment_model=optimal_xgb_treatment,
    control_model=optimal_xgb_control
)
tlearner_xgb.fit(X_train, y_train, t_train)

tlearner_ebm = CustomTLearner(
    treatment_model=optimal_ebm_treatment,
    control_model=optimal_ebm_control
)
tlearner_ebm.fit(X_train, y_train, t_train)

# Serialise to disk — Members 4 and 5 load these, not re-train
joblib.dump(tlearner_xgb, os.path.join(MODELS_DIR, 'tlearner_xgb.pkl'))
joblib.dump(tlearner_ebm, os.path.join(MODELS_DIR, 'tlearner_ebm.pkl'))
print('Models saved to /models/')

In [ ]:
# ============================================================
# 2e. Calibration check  (Member 3 deliverable)
# ============================================================
# A well-calibrated model means: when it says P=0.7, roughly
# 70% of those users actually converted.
# We check calibration on the BASE models (not uplift) to confirm
# the raw probabilities going INTO τ(X) = μ₁ − μ₀ are trustworthy.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, model, name in [
    (axes[0], optimal_ebm_treatment, 'EBM μ₁ (treated)'),
    (axes[1], optimal_xgb_treatment, 'XGB μ₁ (treated)')
]:
    # Predict on the treated TEST users only
    mask = (t_test == 1)
    probs = model.predict_proba(X_test[mask])[:, 1]
    fraction_pos, mean_pred = calibration_curve(y_test[mask], probs, n_bins=10)

    ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    ax.plot(mean_pred, fraction_pos, marker='o', label=name)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_title(f'Calibration — {name}')
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'calibration_plot.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Closer to the diagonal = better calibration.')

## 3 · Member 4 — SHAP explainability
We use the **SHAP difference method** — TreeExplainer on each sub-model
separately, then subtract.  This gives causal-level feature attribution
while keeping EBM's glass-box properties intact.

In [ ]:
# Compute SHAP difference values for the EBM T-Learner
# shap_values_diff() is defined in custom_tlearner.py
print('Computing SHAP values (EBM champion)...')
shap_diff_ebm = tlearner_ebm.shap_values_diff(X_test)
print(f'SHAP diff shape: {shap_diff_ebm.shape}')  # should be (n_test, n_features)

In [ ]:
# ── Global summary: beeswarm plot ─────────────────────────────────────────
# Each dot = one user.
# Position on X axis = how much that feature pushed uplift up or down.
# Color = feature value (red=high, blue=low).
# Features are ranked by mean |SHAP| (most impactful at top).

plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_diff_ebm,
    X_test,
    feature_names=feature_names,
    plot_type='dot',    # beeswarm
    show=False
)
plt.title('SHAP beeswarm — causal uplift attribution (EBM T-Learner)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'shap_beeswarm.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── Global summary: bar plot ──────────────────────────────────────────────
# Cleaner version — shows mean |SHAP| per feature.
# Easier for non-technical audience (e.g., viva panel).

plt.figure(figsize=(9, 6))
shap.summary_plot(
    shap_diff_ebm,
    X_test,
    feature_names=feature_names,
    plot_type='bar',
    show=False
)
plt.title('SHAP bar — mean absolute uplift attribution per feature', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'shap_bar.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── Local explanations: waterfall plots for 3 specific users ─────────────
# Select: one high-uplift user, one low-uplift, one near-zero
uplift_test = tlearner_ebm.predict_uplift(X_test)

idx_high  = np.argmax(uplift_test)                        # most persuadable
idx_low   = np.argmin(uplift_test)                        # sleeping dog
idx_zero  = np.argmin(np.abs(uplift_test))                # unaffected

for idx, label in [(idx_high, 'high_uplift'), (idx_low, 'low_uplift'), (idx_zero, 'near_zero')]:
    # Build a SHAP Explanation object for the waterfall plot
    explanation = shap.Explanation(
        values=shap_diff_ebm[idx],
        base_values=0,           # baseline is zero uplift (no ad effect)
        data=X_test.iloc[idx].values,
        feature_names=feature_names
    )
    plt.figure(figsize=(10, 5))
    shap.waterfall_plot(explanation, show=False)
    plt.title(f'SHAP waterfall — {label} user (predicted uplift={uplift_test[idx]:.4f})', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'shap_waterfall_{label}.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print(f'{label}: τ(X) = {uplift_test[idx]:.4f}')

## 4 · Member 5 — Evaluation: Qini curve + Uplift@K + Bootstrap CI

In [ ]:
# ============================================================
# 4a. Generate uplift predictions on the HELD-OUT test set
# ============================================================
uplift_xgb = tlearner_xgb.predict_uplift(X_test)
uplift_ebm = tlearner_ebm.predict_uplift(X_test)

print(f'XGB uplift — mean: {uplift_xgb.mean():.4f}  std: {uplift_xgb.std():.4f}')
print(f'EBM uplift — mean: {uplift_ebm.mean():.4f}  std: {uplift_ebm.std():.4f}')

# Distribution of uplift scores (sanity check: should be centred near 0)
plt.figure(figsize=(9, 4))
plt.hist(uplift_xgb, bins=60, alpha=0.5, label='XGB T-Learner', density=True)
plt.hist(uplift_ebm, bins=60, alpha=0.5, label='EBM T-Learner', density=True)
plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
plt.xlabel('Predicted uplift τ(X)'); plt.ylabel('Density')
plt.title('Distribution of predicted uplift scores')
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'uplift_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 4b. Qini curve  (scikit-uplift)
# ============================================================
# The Qini curve plots: if we target the top-K% of users ranked
# by predicted uplift, how much INCREMENTAL conversion do we get?
# Area Under the Qini Curve (AUQC) > 0 means our model does better
# than random targeting.

from sklift.viz import plot_qini_curve
from sklift.metrics import qini_auc_score

fig, ax = plt.subplots(figsize=(9, 6))

plot_qini_curve(y_test, uplift_xgb, t_test, label='XGB T-Learner (baseline)', ax=ax)
plot_qini_curve(y_test, uplift_ebm, t_test, label='EBM T-Learner (champion)', ax=ax)

ax.set_title('Qini curve — EBM vs XGBoost T-Learner', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'qini_curve.png'), dpi=300, bbox_inches='tight')
plt.show()

auqc_xgb = qini_auc_score(y_test, uplift_xgb, t_test)
auqc_ebm = qini_auc_score(y_test, uplift_ebm, t_test)
print(f'AUQC — XGB T-Learner : {auqc_xgb:.4f}')
print(f'AUQC — EBM T-Learner : {auqc_ebm:.4f}')

In [ ]:
# ============================================================
# 4c. Bootstrapped Qini — confidence intervals
# ============================================================
# Re-sample the test set 50 times and recompute AUQC each time.
# This gives us a distribution of AUQC scores and lets us plot
# confidence bands around the Qini curve.
# A single-point AUQC could be a lucky draw from the test set —
# the CI tells us how stable the result is.

N_BOOTSTRAP = 50
rng = np.random.default_rng(GLOBAL_SEED)  # reproducible bootstrap

auqc_boot_xgb = []
auqc_boot_ebm = []

n_test = len(y_test)

for i in range(N_BOOTSTRAP):
    # Sample with replacement (same size as test set)
    boot_idx = rng.integers(0, n_test, size=n_test)

    auqc_boot_xgb.append(
        qini_auc_score(y_test[boot_idx], uplift_xgb[boot_idx], t_test[boot_idx])
    )
    auqc_boot_ebm.append(
        qini_auc_score(y_test[boot_idx], uplift_ebm[boot_idx], t_test[boot_idx])
    )

auqc_boot_xgb = np.array(auqc_boot_xgb)
auqc_boot_ebm = np.array(auqc_boot_ebm)

print(f'XGB AUQC — {auqc_xgb:.4f}  95% CI: [{np.percentile(auqc_boot_xgb,2.5):.4f}, {np.percentile(auqc_boot_xgb,97.5):.4f}]')
print(f'EBM AUQC — {auqc_ebm:.4f}  95% CI: [{np.percentile(auqc_boot_ebm,2.5):.4f}, {np.percentile(auqc_boot_ebm,97.5):.4f}]')

# Plot bootstrap distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(auqc_boot_xgb, bins=20, alpha=0.6, label='XGB AUQC bootstrap', color='steelblue')
ax.hist(auqc_boot_ebm, bins=20, alpha=0.6, label='EBM AUQC bootstrap', color='darkorange')
ax.axvline(auqc_xgb, color='steelblue', linestyle='--')
ax.axvline(auqc_ebm, color='darkorange', linestyle='--')
ax.set_xlabel('AUQC score'); ax.set_ylabel('Frequency')
ax.set_title(f'Bootstrap AUQC distribution (n={N_BOOTSTRAP})')
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'qini_bootstrap.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 4d. Uplift@K  —  the business metric
# ============================================================
# If we target the top K% of users by predicted uplift,
# what incremental conversion rate do we achieve?
# This is what a marketing team actually cares about —
# it translates directly to budget ROI.

def uplift_at_k(y_true, uplift_scores, treatment, k_frac):
    """
    Computes the average incremental conversion rate when targeting
    the top k_frac fraction of users ranked by uplift_scores.

    Formula:
        Uplift@K = mean(y | top-K, T=1) − mean(y | top-K, T=0)

    Args:
        y_true        : binary target array
        uplift_scores : predicted uplift τ(X)
        treatment     : binary treatment flag
        k_frac        : fraction of population to target (e.g. 0.1 = top 10%)

    Returns:
        float — incremental conversion rate in the top-K group
    """
    n = len(y_true)
    k = int(n * k_frac)

    # Rank users by descending predicted uplift
    top_k_idx = np.argsort(uplift_scores)[::-1][:k]

    y_topk  = y_true[top_k_idx]
    t_topk  = treatment[top_k_idx]

    treated_conv = y_topk[t_topk == 1].mean() if (t_topk == 1).any() else 0
    control_conv = y_topk[t_topk == 0].mean() if (t_topk == 0).any() else 0

    return treated_conv - control_conv


# Evaluate at 10%, 20%, 30% targeting thresholds
K_VALUES = [0.10, 0.20, 0.30]
results = []

for k in K_VALUES:
    u_xgb = uplift_at_k(y_test, uplift_xgb, t_test, k)
    u_ebm = uplift_at_k(y_test, uplift_ebm, t_test, k)
    results.append({'K': f'Top {int(k*100)}%', 'XGB Uplift@K': u_xgb, 'EBM Uplift@K': u_ebm})
    print(f'Top {int(k*100)}%  — XGB: {u_xgb:.4f}  |  EBM: {u_ebm:.4f}')

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(RESULTS_DIR, 'uplift_at_k.csv'), index=False)
print('\nUplift@K saved to reports/')

In [ ]:
# ============================================================
# 4e. Final summary table — the one slide that wins the viva
# ============================================================

summary = pd.DataFrame({
    'Metric': [
        'AUQC',
        'AUQC 95% CI lower', 'AUQC 95% CI upper',
        'Uplift@10%', 'Uplift@20%', 'Uplift@30%'
    ],
    'XGB T-Learner': [
        auqc_xgb,
        np.percentile(auqc_boot_xgb, 2.5), np.percentile(auqc_boot_xgb, 97.5),
        *[uplift_at_k(y_test, uplift_xgb, t_test, k) for k in K_VALUES]
    ],
    'EBM T-Learner': [
        auqc_ebm,
        np.percentile(auqc_boot_ebm, 2.5), np.percentile(auqc_boot_ebm, 97.5),
        *[uplift_at_k(y_test, uplift_ebm, t_test, k) for k in K_VALUES]
    ]
})

summary = summary.round(4)
print(summary.to_string(index=False))
summary.to_csv(os.path.join(RESULTS_DIR, 'final_metrics_summary.csv'), index=False)